## Getting data from the Guardian API
In this excersice we are going to use the Guardian newspaper API to retrieve news articles about different topics. Each group should decide a topic they want to retrieve. For example: Climate change.

For this excersise you will need an API key from the guardian which you can create here: https://open-platform.theguardian.com/

In [1]:
! pip3 install python-dotenv newspaper3k lxml_html_clean

ERROR: Exception:
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.9/site-packages/pip/_internal/cli/base_command.py", line 107, in _run_wrapper
    status = _inner_run()
  File "/opt/anaconda3/lib/python3.9/site-packages/pip/_internal/cli/base_command.py", line 98, in _inner_run
    return self.run(options, args)
  File "/opt/anaconda3/lib/python3.9/site-packages/pip/_internal/cli/req_command.py", line 96, in wrapper
    return func(self, options, args)
  File "/opt/anaconda3/lib/python3.9/site-packages/pip/_internal/commands/install.py", line 487, in run
    if summary := installed_packages_summary(installed, env):
  File "/opt/anaconda3/lib/python3.9/site-packages/pip/_internal/commands/install.py", line 636, in installed_packages_summary
    installed_versions[distribution.canonical_name] = distribution.version
  File "/opt/anaconda3/lib/python3.9/site-packages/pip/_internal/metadata/pkg_resources.py", line 189, in version
    return parse_version(self._dist.ve

In [2]:
# Libraries we are going to need.
import os
import requests
from time import sleep
from dotenv import load_dotenv
# pip install requests newspaper3k beautifulsoup4 AND pip install lxml_html_clean
from newspaper import Article
from datetime import date, timedelta
from bs4 import BeautifulSoup
from collections import defaultdict
import csv

In [3]:
# List of query words
COUNTRIES = [
    # Major powers / G20
    "united_states", "china", "russia", "india", "germany", "france",
    "united_kingdom", "japan", "italy", "canada", "brazil", "australia",
    "south_korea", "saudi_arabia", "mexico", "indonesia", "turkey",
    "argentina", "south_africa",

    # Europe (expanded)
    "ukraine", "poland", "netherlands", "belgium", "sweden", "norway",
    "denmark", "finland", "switzerland", "austria", "spain", "portugal",
    "greece", "hungary", "czech_republic", "romania", "bulgaria",
    "serbia", "ireland",

    # Middle East
    "israel", "iran", "iraq", "syria", "lebanon", "jordan",
    "united_arab_emirates", "qatar", "kuwait", "oman", "yemen",

    # Africa
    "nigeria", "ethiopia", "egypt", "kenya", "south_africa",
    "ghana", "morocco", "algeria", "tunisia", "sudan",
    "democratic_republic_of_congo", "uganda", "tanzania",

    # Asia
    "pakistan", "bangladesh", "vietnam", "thailand", "philippines",
    "malaysia", "singapore", "taiwan", "north_korea",

    # Americas
    "chile", "peru", "colombia", "venezuela", "cuba",
    "bolivia", "ecuador", "paraguay", "uruguay",

    # Special / geopolitical
    "palestine", "kosovo", "hong_kong"
]

LEADERS = [
    # Major powers
    "joe_biden", "donald_trump", "xi_jinping", "vladimir_putin",
    "narendra_modi", "emmanuel_macron", "olaf_scholz",
    "keir_starmer", "rishi_sunak", "justin_trudeau",

    # Europe
    "volodymyr_zelenskyy", "ursula_von_der_leyen",
    "giorgia_meloni", "mark_rutte",

    # Middle East
    "benjamin_netanyahu", "ali_khamenei", "mohammed_bin_salman",
    "bashar_al_assad",

    # Asia
    "kim_jong_un", "lai_ching_te",

    # Americas
    "lula_da_silva", "javier_milei", "andres_manuel_lopez_obrador",

    # Africa
    "bola_tinubu", "abiy_ahmed",

    # International org leaders
    "antonio_guterres", "klaus_schwab"
]

EVENTS = [
    # Wars / conflicts
    "ukraine_war", "russia_ukraine_war", "gaza_war",
    "israel_hamas_war", "syrian_civil_war",

    # Political processes
    "brexit", "election", "midterm_election", "general_election",
    "referendum",

    # Global crises
    "covid_19", "pandemic", "inflation", "recession",
    "energy_crisis", "climate_change",

    # Policy / governance
    "sanctions", "trade_war", "immigration", "border_policy",
    "foreign_policy", "defense_policy",

    # Tech / geopolitics
    "artificial_intelligence", "ai_regulation",
    "cybersecurity", "surveillance",

    # Economy
    "global_economy", "interest_rates", "central_bank",
    "supply_chain", "oil_prices"
]

INSTITUTIONS = [
    "nato", "united_nations", "eu", "european_union",
    "world_bank", "imf", "world_health_organization",
    "wto", "g7", "g20", "brics",
    "white_house", "kremlin", "downing_street",
    "pentagon", "european_commission"
]

QUERY_TERMS = COUNTRIES + LEADERS + EVENTS + INSTITUTIONS

In [4]:
# Helpers
def get_article_text(url):
    try:
        article = Article(url)
        article.download()
        article.parse()
        return article.text
    except:
        return None

In [5]:
# Getting the API key and creating the endpoint
load_dotenv()
API_KEY = os.getenv("GUARDIAN_API_KEY")

guardian_endpoint = "https://content.guardianapis.com/search"

In [8]:
articles_by_year = defaultdict(list)

articles_by_year = defaultdict(list)
seen_urls = set()  # avoid duplicates

for year in range(2024, 2025):
    for month in range(1, 13):

        # stop at Apr 2026
        if year == 2026 and month > 4:
            break

        start_date = f"{year}-{month:02d}-01"

        if month == 12:
            end_date = f"{year+1}-01-01"
        else:
            end_date = f"{year}-{month+1:02d}-01"

        print(f"Fetching {start_date} to {end_date}")

        page = 1

        while True:
            parameters = {
                "api-key": API_KEY,

                # BROAD QUERY (or remove entirely)
                # "q": "politics OR government OR world",

                # BETTER: use sections instead of narrow keyword
                "section": "world|politics|business",

                "from-date": start_date,
                "to-date": end_date,
                "page-size": 200,
                "page": page,
                "order-by": "newest",
            }

            response = requests.get(guardian_endpoint, params=parameters)
            response.raise_for_status()
            data = response.json()

            results = data.get("response", {}).get("results", [])

            if not results:
                break

            for article in results:
                title = article["webTitle"]
                date = article["webPublicationDate"]
                url = article["webUrl"]

                # avoid duplicates
                if url in seen_urls:
                    continue
                seen_urls.add(url)

                # get full text
                text = get_article_text(url)

                # basic quality filter
                if text and len(text.split()) > 200:
                    articles_by_year[str(year)].append(
                        [title, date, url, text]
                    )

            if page >= data["response"]["pages"]:
                break

            page += 1
            sleep(1)  # respect API rate limits

# Save CSV files
for year, articles in articles_by_year.items():
    filename = f"guardian_{year}.csv"
    with open(filename, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["Title", "Publication Date", "URL"])
        writer.writerows(articles)

    print(f"Saved {filename} ({len(articles)} articles)") 

# Save CSV files
for year, articles in articles_by_year.items():
    filename = f"guardian_{year}.csv"
    with open(filename, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["Title", "Publication Date", "URL", "Text"])
        writer.writerows(articles)

    print(f"Saved {filename} ({len(articles)} articles)")

Fetching 2024-01-01 to 2024-02-01
Fetching 2024-02-01 to 2024-03-01
Fetching 2024-03-01 to 2024-04-01
Fetching 2024-04-01 to 2024-05-01
Fetching 2024-05-01 to 2024-06-01
Fetching 2024-06-01 to 2024-07-01
Fetching 2024-07-01 to 2024-08-01
Fetching 2024-08-01 to 2024-09-01
Fetching 2024-09-01 to 2024-10-01
Fetching 2024-10-01 to 2024-11-01
Fetching 2024-11-01 to 2024-12-01
Fetching 2024-12-01 to 2025-01-01
Saved guardian_2024.csv (10208 articles)
Saved guardian_2024.csv (10208 articles)
